In [1]:
import pandas as pd
import re

In [2]:
df1 = pd.read_csv("db_drug_interactions.csv")
df2 = pd.read_csv("medicines_complete.csv")

In [6]:
df1.head()
df1.shape

(191541, 3)

In [7]:
df2.head()
df2.shape

(251045, 19)

In [8]:
df1_original = df1.copy()

In [9]:
def extract_salts(s):
    if pd.isna(s):
        return []
    
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)  # remove dosage like (500mg)
    
    return [x.strip() for x in s.split("+")]

df2["salt_list"] = df2["Salt_Composition"].apply(extract_salts)

In [10]:
df2_exp = df2.explode("salt_list")
df2_exp = df2_exp.rename(columns={"salt_list": "drug"})

# remove empty values if any
df2_exp = df2_exp[df2_exp["drug"].notna()]

In [11]:
df1["Drug 1"] = df1["Drug 1"].str.lower().str.strip()
df1["Drug 2"] = df1["Drug 2"].str.lower().str.strip()

In [12]:
drug_to_products = (
    df2_exp.groupby("drug")["Brand"]
    .apply(list)
    .to_dict()
)

In [13]:
df1["Drug1_Products"] = df1["Drug 1"].map(drug_to_products)
df1["Drug2_Products"] = df1["Drug 2"].map(drug_to_products)

In [14]:
print("Original rows:", len(df1_original))
print("Final rows   :", len(df1))

Original rows: 191541
Final rows   : 191541


In [15]:
print(df1[["Drug1_Products", "Drug2_Products"]].isna().mean())

Drug1_Products    0.401449
Drug2_Products    0.392433
dtype: float64


In [16]:
test_drug = "amoxycillin"

print("From mapping:", drug_to_products.get(test_drug))
print("From df2_exp:", df2_exp[df2_exp["drug"] == test_drug]["Brand"].unique())

From mapping: ['Augmentin 625 Duo Tablet', 'Amoxyclav 625 Tablet', 'Augmentin Duo Oral Suspension', 'Almox 500 Capsule', 'Augmentin DDS Suspension', 'Amoxycillin 500mg Capsule', 'Augmentin 1000 Duo Tablet', 'Augmentin 375 Tablet', 'Augmentin 1.2gm Injection', 'Advent Forte 457mg Syrup Tangy Orange', 'Advent 228.5mg Dry Syrup', 'Almox-CV 625 Tablet', 'Advent 625 Tablet', 'Amox 500mg Capsule', 'Almox 250 Capsule', 'Augmentin ES Oral Suspension', 'Amoxyclav 375 Tablet', 'Augpen -DS Suspension', 'Augpen 625 BID Tablet', 'Augpen LB 625 Tablet', 'Amoxycillin 125mg Oral Suspension', 'Advent 91.4mg Drops Tangy Orange', 'Amoxyclav DS Syrup', 'Amyclox-LB Capsule', 'Amoxycillin  250mg Capsule', 'Amoxy 500mg Capsule', 'Amoxclav 500 mg/125 mg Tablet', 'Amoxyclav Oral Suspension', 'Almox 250 mg Tablet DT', 'Augpen HS 200 mg/28.5 mg Suspension', 'Advent 457mg Tablet DT', 'Amoxible-CL 625 Tablet', 'Ampilox CV 500 mg/125 mg Tablet', 'Amoxil 250mg Capsule', 'Amoxy 250mg Capsule', 'Amoxil 250mg Tablet DT

In [17]:
drug_to_product = (
    df2_exp.drop_duplicates("drug")
    .set_index("drug")["Brand"]
)

df1["Drug1_Product"] = df1["Drug 1"].map(drug_to_product)
df1["Drug2_Product"] = df1["Drug 2"].map(drug_to_product)

In [18]:
df1[["Drug 1", "Drug 2", "Drug1_Products", "Drug2_Products"]].head()

,Drug 1,Drug 2,Drug1_Products,Drug2_Products
0,trioxsalen,verteporfin,NaN,[Visudyne Injection]
1,aminolevulinic acid,verteporfin,NaN,[Visudyne Injection]
2,titanium dioxide,verteporfin,NaN,[Visudyne Injection]
3,tiaprofenic acid,verteporfin,NaN,[Visudyne Injection]
4,cyamemazine,verteporfin,NaN,[Visudyne Injection]


In [ ]:
from itertools import product

def generate_pairs(row):
    d1 = row["Drug1_Products"]
    d2 = row["Drug2_Products"]
    
    if isinstance(d1, list) and isinstance(d2, list):
        return list(product(d1, d2))
    return None

df1["Product_Pairs"] = df1.apply(generate_pairs, axis=1)